# Part 1 · Where do people want to go?
### A quick first run: OD weights and where crowds form

In about 15 minutes you run a small station a few times and see one big idea:
**who goes where** matters as much as **how many** people there are.
Work in pairs: one person clicks, the other takes notes.

**How to use this notebook**
* Press **▶** on the left of a grey cell (or click it and press **Shift + Enter**) to run it.
* Cells with sliders and boxes are *forms*: change the values on the right, then press ▶ again.
* You never have to edit code. If you're curious, double-click a form to see the code behind it.
* Something broken? **Runtime → Restart session**, then run the first cell again.

## ▶ Step 1: install (run this first, ≈30 s)
This is how you install **any** Python package from GitHub. `pednstream-workshop` is a small
package with this workshop's network and plotting helpers; installing it also installs
**PedNStream** itself. If Colab restarted, run it again.
You can ignore the note *"you may need to restart the kernel"*.

In [ ]:
%pip install -q "pednstream-workshop @ git+https://github.com/WaimenMak/pednstream-workshop.git"
import json, importlib.metadata as md
if json.loads(md.distribution("pednstream").read_text("direct_url.json") or "{}").get("vcs_info", {}).get("commit_id") != "4a85236b3c47b17f25c2bfe8d5c0ef35651900a7":
    %pip install -q --force-reinstall --no-deps "pednstream @ git+https://github.com/WaimenMak/PedNStream.git@4a85236b3c47b17f25c2bfe8d5c0ef35651900a7"

import matplotlib.pyplot as plt
import pednstream_workshop
if tuple(map(int, pednstream_workshop.__version__.split("."))) < (1, 7, 0):
    raise RuntimeError("An old version of the workshop package is loaded. "
                       "Click Runtime → Restart session, then run this cell again.")
from pednstream_workshop import run, compare, compare_maps
plt.rcParams["figure.dpi"] = 90
print("✅ Ready! Scroll down and continue.")

## 1 · Meet *Station Square*
```
            6  City Exit (Y)
            |
            4  Mall Junction
            |  \
   (wide)   |    \   (narrow diagonal)
            |      \
   0 ───────2 ───────3 ─────────── 5  Platform (X)
  West     Plaza  ↑  Hall
  Gate (A)    NARROW  |
                      1  South Gate (B)
```
* **Two entrances**: West Gate **A** enters at the Plaza, South Gate **B** enters at the Hall.
* **Two destinations**: the **Platform X** and the **City Exit Y**.
* Corridor **2_3** (Plaza–Hall) is narrow: only **2 m** wide.
* A rush of people arrives at both gates, peaking around minute 17. The rush is **the same in every run**.
* Where there are two routes, people choose by **distance**, **crowding** and **width**.

## 2 · What is an OD weight?
An **OD weight** says how strongly the people entering at a gate (the *origin*) are drawn to each
*destination*. Weights are **relative**: `3` and `1` mean "three times as many head to the Platform as
to the City Exit". The gate's demand sets how *many* people enter; the weights only decide where they head.

The cell below opens **only Gate A**. Press ▶, then move the sliders and press ▶ again.

In [ ]:
#@title ▶ Try OD weights for Gate A {display-mode: "form"}
weight_A_to_Platform = 3  #@param {type:"slider", min:0, max:10, step:1}
weight_A_to_CityExit = 1  #@param {type:"slider", min:0, max:10, step:1}
only_A = run(od={"0_5": weight_A_to_Platform, "0_6": weight_A_to_CityExit}, origins=[0])

od = only_A.network.od_tracker.od_table()        # PedNStream follows every person from gate to destination
to_x, to_y = od[(0, 5)]["entered"], od[(0, 6)]["entered"]
share = to_x / max(to_x + to_y, 1e-9)
print(f"Gate A sent {share:.0%} to Platform X ({to_x:.0f} people) "
      f"and {1 - share:.0%} to City Exit Y ({to_y:.0f} people)")

> **Try:** 1 & 1, then 10 & 10. Same or different? Then 1 & 0.
>
> Every person who enters gets a destination drawn from the weights and keeps it all the way.
> **The weights decide where people go; the simulation decides which route they take and when they arrive.**

---
## 3 · Same totals, different mix ⭐
Now both gates are open. Three ways to divide the travellers, and in all three **half of all travellers
go to each destination**. Only *who* goes where differs:

| scenario | Gate A goes to | Gate B goes to |
|---|---|---|
| **baseline** | half X, half Y | half X, half Y |
| **crossing** | Platform X (east) | City Exit Y (north) |
| **parallel** | City Exit Y (north) | Platform X (east) |

Guess first with your partner: will the three behave the same? Then press ▶.

In [ ]:
#@title ▶ Run the three scenarios {display-mode: "form"}
base = run()
crossing = run(od={"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1})
parallel = run(od={"0_5": 0, "0_6": 1, "1_5": 1, "1_6": 0})
runs = {"baseline": base, "crossing": crossing, "parallel": parallel}
compare_maps(runs); plt.show()
display(compare(runs))

**How to read this**
* **Map**: corridor colour = the highest density (people per m²) during the run. Above **2** (yellow,
  then orange and red) people slow down sharply.
* **time spent inside (person-hours)**: the main "how bad was it" number. **Lower is better.**
* **turned away**: people who could not even enter because the first corridor was full.

**Discuss with your partner**
1. *reached Platform X* is about the same in all three, so why does one scenario cost so much more time?
   *(Hint: compare corridor 2_3 on the crossing and parallel maps. In the crossing scenario Gate A's
   walkers cross it eastwards to the Platform. Which way do Gate B's walkers to the City Exit go?)*
2. Knowing only "50% go to the Platform", could you have predicted this?

---
## Take-away
* An OD weight sets **where** people want to go, not **how many** there are.
* Two OD patterns with the same destination totals can behave very differently: when routes **overlap**
  and walkers meet **head-on**, corridors jam.

➡ Next: **Part 2 · Playground**: change the OD weights, the demand, how much people avoid crowds and
the width of corridor 2_3 yourself. Then **Part 3 · Challenge**.